# Colab: 현재 이미지·라벨만 압축 해제하여 검수
Drive에서 이 노트북과 ZIP을 같은 폴더에 두고 실행합니다.
전체 이미지를 풀지 않습니다. **현재 보는 이미지와 TXT 라벨 한 쌍만 Colab 임시 폴더에 추출**합니다.
다른 이미지로 이동하면 이전 임시 파일을 지우고 다음 한 쌍만 추출합니다.

삭제 버튼은 현재 임시 파일을 지우고 **Drive의 삭제 목록**에 기록해 이후 검수에서 즉시 제외합니다.
원본 ZIP은 검수 중 변경하지 않습니다. 마지막 셀에서 삭제 항목을 제외한 새 ZIP을 저장합니다.
원본 ZIP은 시작할 때 Colab 로컬로 한 번 복사합니다. 새 ZIP도 로컬에서 파일 하나씩 처리해 생성·검사한 뒤 완성된 파일만 Drive로 복사합니다.
로컬 디스크에는 원본 ZIP과 결과 ZIP을 함께 보관할 여유 공간이 필요합니다. 전체 이미지는 압축 해제하지 않습니다.
재접속 후 같은 ZIP과 검수 기록 폴더를 선택하면 삭제 목록을 이어 사용합니다.

## 1. 설치 및 Google Drive 마운트
Colab에서 실행하고 Google Drive 연결을 승인하세요.

In [ ]:
import sys
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "matplotlib", "Pillow", "PyYAML", "ipywidgets"])

import io
import math
from pathlib import Path, PurePosixPath
import zipfile
import yaml
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle
import ipywidgets as widgets
from IPython.display import display, clear_output


import json
import shutil
import tempfile
import hashlib
from datetime import datetime, timezone
from google.colab import drive
drive.mount("/content/drive")


## 2. 노트북이 저장된 Drive 폴더 지정
NOTEBOOK_DIR에 **노트북과 ZIP이 함께 있는 Drive 폴더**를 입력하세요.
예: /content/drive/MyDrive/hand_dataset
Colab의 현재 작업 폴더는 /content이므로 노트북의 Drive 위치를 자동으로 알아낼 수 없습니다.

해당 폴더에 ZIP 하나면 자동 선택하며, 여러 개면 ZIP_NAME에 파일명만 입력합니다.
클래스는 YAML에서 읽습니다. 없다면 CLASS_NAMES_OVERRIDE에 기존 데이터는 {0: "hand"}, 재라벨링은 {0: "left_hand", 1: "right_hand"}를 지정하세요.


In [ ]:
NOTEBOOK_DIR = "/content/drive/MyDrive/hand_dataset"  # 노트북과 ZIP이 있는 폴더로 수정
ZIP_NAME = ""  # 같은 폴더에 ZIP이 여러 개면 파일명 입력
CLASS_NAMES_OVERRIDE = None
NOTEBOOK_DIR = Path(NOTEBOOK_DIR).expanduser().resolve()
DRIVE_ROOT = Path("/content/drive").resolve()
if not NOTEBOOK_DIR.is_relative_to(DRIVE_ROOT) or not NOTEBOOK_DIR.is_dir():
    raise ValueError("Drive를 마운트하고 NOTEBOOK_DIR을 실제 Drive 폴더로 지정하세요.")
print("ZIP 탐색 폴더:", NOTEBOOK_DIR)
zip_candidates = sorted(p for p in NOTEBOOK_DIR.iterdir()
                        if p.is_file() and p.suffix.lower() == ".zip")
if ZIP_NAME:
    if Path(ZIP_NAME).name != ZIP_NAME or "/" in ZIP_NAME or "\\" in ZIP_NAME or ":" in ZIP_NAME:
        raise ValueError("ZIP_NAME에는 같은 폴더의 파일명만 입력하세요.")
    chosen_zip = NOTEBOOK_DIR / ZIP_NAME
else:
    if len(zip_candidates) != 1:
        raise ValueError("같은 폴더에 ZIP 하나를 놓거나 ZIP_NAME을 지정하세요. 발견: "
                         + ", ".join(p.name for p in zip_candidates))
    chosen_zip = zip_candidates[0]
if chosen_zip.resolve().parent != NOTEBOOK_DIR or chosen_zip.suffix.lower() != ".zip" or not chosen_zip.is_file():
    raise ValueError("노트북 폴더 안의 ZIP 파일이어야 합니다.")
DATA_PATH = str(chosen_zip)
print("선택한 ZIP:", chosen_zip.name)

# 셀 재실행 시 이전 ZIP 핸들을 닫습니다.
if "archive" in globals() and archive is not None:
    archive.close()
archive = None
if "source_cache" in globals():
    source_cache.cleanup()
source_cache = tempfile.TemporaryDirectory(prefix="hand_review_source_")
local_zip = Path(source_cache.name) / "source.zip"
print("원본 ZIP을 Colab 로컬로 복사 중...")
try:
    shutil.copyfile(chosen_zip, local_zip)
except Exception:
    source_cache.cleanup()
    raise
print("로컬 복사 완료:", local_zip.stat().st_size, "bytes")
dataset_path = Path(DATA_PATH).expanduser().resolve()
if dataset_path.is_file() and zipfile.is_zipfile(local_zip):
    archive = zipfile.ZipFile(local_zip)
    info = [entry for entry in archive.infolist() if not entry.is_dir()]
    file_names = {entry.filename for entry in info}
    if len(file_names) != len(info):
        raise ValueError("ZIP에 중복 파일명이 있습니다.")
elif dataset_path.is_dir():
    file_names = {p.relative_to(dataset_path).as_posix()
                  for p in dataset_path.rglob("*")
                  if p.is_file() and p.resolve().is_relative_to(dataset_path)}
else:
    raise ValueError(f"ZIP 또는 폴더가 아닙니다: {dataset_path}")

def read_bytes(name):
    if name not in file_names:
        raise FileNotFoundError(name)
    if archive is not None:
        return archive.read(name)
    path = (dataset_path / name).resolve()
    if not path.is_relative_to(dataset_path):
        raise ValueError("데이터 폴더 밖의 경로입니다.")
    return path.read_bytes()

candidates = []
for name in sorted(file_names):
    p = PurePosixPath(name)
    if p.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp", ".webp"}:
        continue
    for i in range(len(p.parts) - 2):
        if p.parts[i] == "images" and p.parts[i+1] in {"train", "val", "test"}:
            candidates.append((PurePosixPath(*p.parts[:i]), p, i))
roots = {root for root, _, _ in candidates}
if len(roots) != 1:
    raise ValueError("images/train, images/val 등이 있는 데이터 루트 하나가 필요합니다.")
root = roots.pop()
items = []
for _, image, index in candidates:
    label = root / "labels" / PurePosixPath(*image.parts[index+1:]).with_suffix(".txt")
    items.append({"image": image.as_posix(), "label": label.as_posix(),
                  "split": image.parts[index+1],
                  "relative": image.relative_to(root).as_posix()})

class_names = {}
yaml_candidates = sorted(n for n in file_names
                         if PurePosixPath(n).parent == root
                         and PurePosixPath(n).suffix.lower() in {".yaml", ".yml"})
if CLASS_NAMES_OVERRIDE is not None:
    class_names = {int(k): str(v) for k, v in CLASS_NAMES_OVERRIDE.items()}
else:
    for name in yaml_candidates:
        config = yaml.safe_load(read_bytes(name).decode("utf-8-sig"))
        if not isinstance(config, dict) or "names" not in config:
            continue
        names = config["names"]
        mapping = ({int(k): str(v) for k, v in names.items()} if isinstance(names, dict)
                   else dict(enumerate(map(str, names))))
        if class_names and mapping != class_names:
            raise ValueError("YAML마다 클래스 이름이 다릅니다. CLASS_NAMES_OVERRIDE를 지정하세요.")
        class_names = mapping
print("이미지:", len(items), "| 클래스:", class_names or "없음: class_N으로 표시")
print("분할:", {s: sum(i["split"] == s for i in items) for s in ("train", "val", "test")})

## 3. 검수 기록과 한 장 캐시 준비
STATE_DIR에는 review_state.json만 저장합니다. 기본값은 같은 Drive 폴더의 review_state/ZIP이름/입니다.
현재 이미지·라벨 캐시는 Colab 로컬 임시 폴더에 둡니다. 이 단계에서는 이미지 파일을 읽거나 압축 해제하지 않습니다.
이전 전체 추출 방식의 작업 폴더와 별개이며, 기존 파일은 자동 삭제하지 않습니다.

In [ ]:
STATE_DIR = str(NOTEBOOK_DIR / "review_state" / dataset_path.stem)

class ReviewWorkspace:
    def __init__(self, folder, entries, names, reader, available, source_id, source_root=".", opener=None):
        self.folder = Path(folder).expanduser().resolve()
        self.state_path = self.folder / "review_state.json"
        self.reader, self.opener = reader, opener
        self.available = set(available)
        self.source_root = PurePosixPath(source_root)
        self.sources = {}
        normalized, used_labels = [], set()
        for entry in entries:
            image = entry["relative"]
            label = (PurePosixPath("labels") / PurePosixPath(*PurePosixPath(image).parts[1:]).with_suffix(".txt")).as_posix()
            if label in used_labels:
                raise ValueError("이미지들이 같은 라벨을 공유합니다: " + label)
            used_labels.add(label)
            self.sources[image], self.sources[label] = entry["image"], entry["label"]
            normalized.append(dict(image=image, label=label, relative=image, split=entry["split"]))
        if self.state_path.exists():
            self.state = json.loads(self.state_path.read_text(encoding="utf-8"))
            if self.state.get("version") != 3 or self.state["source_id"] != source_id:
                raise ValueError("다른 데이터/이전 형식의 기록입니다. 새 STATE_DIR을 지정하세요.")
        else:
            self.folder.mkdir(parents=True, exist_ok=True)
            self.state = dict(version=3, source_id=source_id, names=names, entries=normalized, deleted=[])
            self.save()
        self.cache = Path(tempfile.mkdtemp(prefix="hand_review_current_")).resolve()
        self.cached = {}
        self.current = None

    @staticmethod
    def safe_path(base, relative):
        part = PurePosixPath(relative)
        if part.is_absolute() or ".." in part.parts or "\\" in relative or ":" in relative:
            raise ValueError("안전하지 않은 상대 경로")
        path = base.joinpath(*part.parts).resolve()
        if not path.is_relative_to(base.resolve()):
            raise ValueError("임시 폴더 외부 경로")
        return path

    def save(self):
        partial = self.state_path.with_suffix(".tmp")
        partial.write_text(json.dumps(self.state, ensure_ascii=False, indent=2), encoding="utf-8")
        partial.replace(self.state_path)

    def remaining(self):
        deleted = {record["image"] for record in self.state["deleted"]}
        return [entry for entry in self.state["entries"] if entry["image"] not in deleted]

    def exists(self, relative):
        return self.sources.get(relative) in self.available

    def clear_cache(self):
        for path in self.cached.values():
            # 고정 임시 폴더 내부의 이번 인스턴스가 만든 파일만 삭제합니다.
            if not path.resolve().is_relative_to(self.cache):
                raise ValueError("캐시 외부 경로")
            path.unlink(missing_ok=True)
        self.cached = {}
        self.current = None

    def prepare(self, entry):
        if entry not in self.remaining():
            raise ValueError("현재 데이터의 활성 이미지가 아닙니다.")
        if self.current == entry["image"]:
            return
        self.clear_cache()
        try:
            for key in ("image", "label"):
                relative = entry[key]
                if not self.exists(relative):
                    continue
                # 파일은 두 개뿐이며 원본 이름을 경로로 사용하지 않습니다.
                suffix = PurePosixPath(relative).suffix
                dest = self.cache / (key + suffix)
                self.cached[relative] = dest
                with dest.open("wb") as output:
                    self.copy_member(self.sources[relative], output)
            self.current = entry["image"]
        except Exception:
            self.clear_cache()
            raise

    def read_current(self, relative):
        if relative not in self.cached:
            raise FileNotFoundError("현재 이미지의 캐시가 아닙니다: " + relative)
        return self.cached[relative].read_bytes()

    def copy_member(self, name, output):
        if name not in self.available:
            raise FileNotFoundError(name)
        if self.opener is None:
            output.write(self.reader(name))
        else:
            with self.opener(name) as source:
                shutil.copyfileobj(source, output, length=1024*1024)

    def delete(self, entry):
        if entry not in self.remaining():
            raise ValueError("활성 이미지가 아닙니다.")
        record = dict(image=entry["image"], files=[entry["image"], entry["label"]],
                      deleted_utc=datetime.now(timezone.utc).isoformat())
        self.state["deleted"].append(record)
        try:
            self.save()
        except Exception:
            self.state["deleted"].pop()
            raise
        if self.current == entry["image"]:
            self.clear_cache()

    def export(self, target):
        entries = self.remaining()
        if not entries:
            raise ValueError("남은 이미지가 없습니다.")
        target = Path(target).resolve()
        if target.exists():
            raise FileExistsError("새 ZIP 파일명을 사용하세요.")
        partial = target.with_suffix(".part")
        try:
            with tempfile.TemporaryDirectory(prefix="hand_review_export_") as staging:
                local_output = Path(staging) / "dataset.zip"
                print("로컬에서 결과 ZIP 생성 중...", len(entries), "images")
                with zipfile.ZipFile(local_output, "w", compression=zipfile.ZIP_DEFLATED) as output:
                    for entry in entries:
                        for key in ("image", "label"):
                            relative = entry[key]
                            if not self.exists(relative):
                                raise FileNotFoundError("이미지/라벨 쌍이 없습니다: " + relative)
                            with output.open(relative, "w", force_zip64=True) as dest:
                                self.copy_member(self.sources[relative], dest)
                    config = {s:"images/"+s for s in sorted({e["split"] for e in entries})}
                    config.update(names={int(k):v for k,v in self.state["names"].items()},
                                  kpt_shape=[21,3], flip_idx=list(range(21)))
                    output.writestr("data.yaml", yaml.safe_dump(config, sort_keys=False))
                    output.writestr("review_log.json", json.dumps(self.state, ensure_ascii=False, indent=2))
                    for name in sorted(self.available):
                        path = PurePosixPath(name)
                        if path.parent == self.source_root and (
                            path.name.lower().startswith(("readme", "license")) or
                            path.name in ("metadata.json", "environment.txt", "manifest.jsonl")):
                            with output.open("provenance/"+path.name, "w", force_zip64=True) as dest:
                                self.copy_member(name, dest)
                    manifest = (self.source_root/"manifest.jsonl").as_posix()
                    if manifest in self.available:
                        keep = {e["image"] for e in entries}
                        source = self.opener(manifest) if self.opener else io.BytesIO(self.reader(manifest))
                        with source, output.open("manifest.jsonl", "w", force_zip64=True) as dest:
                            for line in source:
                                if not line.strip():
                                    continue
                                row = json.loads(line)
                                if row.get("status") == "accepted" and row.get("image") in keep:
                                    dest.write((json.dumps(row, ensure_ascii=False)+"\n").encode())
                with zipfile.ZipFile(local_output) as check:
                    bad_member = check.testzip()
                    if bad_member is not None:
                        raise ValueError("ZIP 무결성 검사 실패: " + bad_member)
                print("완성된 ZIP을 저장 위치로 복사 중...", local_output.stat().st_size, "bytes")
                shutil.copyfile(local_output, partial)
            partial.replace(target)
        finally:
            partial.unlink(missing_ok=True)
        return target

# ZIP 핸들을 직접 바인딩해 표시용 read_bytes 재정의와 분리합니다.
if "workspace" in globals() and hasattr(workspace, "clear_cache"):
    workspace.clear_cache()
source_entries, source_files = items, set(file_names)
source_id = hashlib.sha256(json.dumps(sorted(
    (i.filename,i.file_size,i.CRC) for i in archive.infolist() if not i.is_dir())).encode()).hexdigest()
workspace = ReviewWorkspace(STATE_DIR, source_entries,
    class_names or {0:"class_0", 1:"class_1"}, archive.read, source_files, source_id,
    source_root=root.as_posix(), opener=archive.open)
class_names = {int(k):v for k,v in workspace.state["names"].items()}

def read_bytes(name):
    return workspace.read_current(name)

def update_items():
    global items, file_names
    items = workspace.remaining()
    file_names = {e[k] for e in items for k in ("image","label") if workspace.exists(e[k])}

update_items()
print("Drive 검수 기록:", workspace.folder)
print("현재 한 쌍 임시 캐시:", workspace.cache)
print("남은 이미지:", len(items), "| 삭제:", len(workspace.state["deleted"]))

## 4. 표시 함수
라벨 오류와 누락도 표시합니다. v는 학습 표기 마스크이며 관절 confidence가 아닙니다.

In [ ]:
EDGES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
         (5,9),(9,10),(10,11),(11,12),(9,13),(13,14),(14,15),(15,16),
         (13,17),(0,17),(17,18),(18,19),(19,20)]
PAGE_SIZE = 1
COLORS = ["#00e5a0", "#ffb347", "#60b7ff", "#e89aff"]

def parse_label(text):
    hands, errors = [], []
    for row, line in enumerate(text.splitlines(), 1):
        if not line.strip():
            continue
        try:
            v = [float(x) for x in line.split()]
            if len(v) != 68 or not all(math.isfinite(x) for x in v):
                raise ValueError("expected 68 finite values")
            if v[0] < 0 or not v[0].is_integer():
                raise ValueError("invalid class ID")
            points = [v[i:i+3] for i in range(5, 68, 3)]
            if any(p[2] not in (0, 1, 2) for p in points):
                raise ValueError("invalid keypoint mask")
            if any(not 0 <= x <= 1 for p in points for x in p[:2]):
                errors.append(f"row {row}: keypoint outside image")
            if not (0 <= v[1] <= 1 and 0 <= v[2] <= 1 and 0 < v[3] <= 1 and 0 < v[4] <= 1):
                errors.append(f"row {row}: invalid bbox")
            hands.append({"class_id": int(v[0]), "bbox": v[1:5], "points": points})
        except ValueError as exc:
            errors.append(f"row {row}: {exc}")
    return hands, errors

def select_page(page=1, split="all", search=""):
    selected = [item for item in items
                if (split == "all" or item["split"] == split)
                and search.lower().strip() in item["relative"].lower()]
    pages = max(1, math.ceil(len(selected) / PAGE_SIZE))
    if not 1 <= page <= pages:
        raise ValueError(f"페이지 범위: 1..{pages}")
    return selected[(page-1)*PAGE_SIZE:page*PAGE_SIZE], pages, len(selected)

def show_page(page=1, split="all", search="", columns=1,
              show_points=True, show_numbers=True, show_boxes=True):
    if columns != 1:
        raise ValueError("columns=1을 사용하세요.")
    batch, pages, total = select_page(page, split, search)
    if not batch:
        print("조건에 맞는 이미지가 없습니다.")
        return
    rows = math.ceil(len(batch)/columns)
    fig, axes = plt.subplots(rows, columns, figsize=(12, 9), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for offset, (ax, item) in enumerate(zip(axes.flat, batch)):
        number = (page-1)*PAGE_SIZE+offset+1
        # 해시 파일명은 제목에서 줄이고 전체 경로는 아래 목록으로 제공합니다.
        basename = PurePosixPath(item["relative"]).name
        short_name = basename if len(basename) <= 36 else basename[:22] + "…" + basename[-8:]
        ax.set_title(f'{number}. [{item["split"]}] {short_name}', fontsize=10)
        try:
            with Image.open(io.BytesIO(read_bytes(item["image"]))) as image:
                rgb = image.convert("RGB")
                width, height = rgb.size
                ax.imshow(rgb)
        except (OSError, ValueError) as exc:
            ax.text(.05, .5, "Image error: " + str(exc), transform=ax.transAxes,
                    fontsize=9, color="red", wrap=True)
            continue
        if item["label"] not in file_names:
            hands, errors = [], ["Missing label"]
        else:
            try:
                hands, errors = parse_label(read_bytes(item["label"]).decode("utf-8-sig"))
                if not hands and not errors:
                    errors = ["Empty label (background)"]
            except (OSError, UnicodeError) as exc:
                hands, errors = [], [str(exc)]
        for hand_index, hand in enumerate(hands):
            color = COLORS[hand_index % len(COLORS)]
            points = [(x*width, y*height, v) for x, y, v in hand["points"]]
            if show_points:
                for a, b in EDGES:
                    if points[a][2] and points[b][2]:
                        ax.plot([points[a][0], points[b][0]], [points[a][1], points[b][1]],
                                color=color, linewidth=1.5)
                ax.scatter([p[0] for p in points if p[2]], [p[1] for p in points if p[2]],
                           c=color, s=16, edgecolors="black", linewidths=.4)
            if show_numbers:
                for j, (x, y, v) in enumerate(points):
                    if v:
                        ax.text(x, y, str(j), fontsize=8, color="white", clip_on=True,
                                path_effects=[pe.withStroke(linewidth=2, foreground="black")])
            cx, cy, bw, bh = hand["bbox"]
            x, y = (cx-bw/2)*width, (cy-bh/2)*height
            if show_boxes:
                ax.add_patch(Rectangle((x,y), bw*width, bh*height, fill=False,
                                       edgecolor=color, linewidth=1.5))
                ax.text(max(0,x), max(0,y), class_names.get(hand["class_id"], f'class_{hand["class_id"]}'),
                        color="black", fontsize=9, va="top", clip_on=True,
                        bbox=dict(facecolor=color, alpha=.85, pad=2))
        # 범위 밖 라벨이 있어도 축이 늘어나 이미지가 작아지지 않게 고정합니다.
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)
        if errors:
            ax.text(.02, .02, "\n".join(errors[:4]), transform=ax.transAxes, fontsize=8,
                    color="white", va="bottom", bbox=dict(facecolor="darkred", alpha=.8))
    fig.suptitle(f"Page {page}/{pages} | {total} images | keypoints: stored labels",
                 fontsize=13)
    fig.tight_layout(rect=(0,0,1,.98))
    plt.show()
    plt.close(fig)
    for offset, item in enumerate(batch):
        print(f'{(page-1)*PAGE_SIZE+offset+1:>5}  {item["relative"]}')

## 5. 한 장씩 검수·삭제
현재 한 쌍만 압축 해제합니다. 이전/다음으로 이동하면 이전 캐시 파일을 지웁니다.
삭제 버튼을 누르면 현재 캐시를 지우고 Drive의 삭제 목록에 즉시 기록한 뒤 다음 이미지를 표시합니다.
ZIP 자체에서의 삭제는 마지막 새 ZIP 저장에 반영됩니다. 검수 중 원본 ZIP을 덮어쓰지 않습니다.

In [ ]:
split_widget=widgets.Dropdown(options=["all"]+sorted({e["split"] for e in workspace.state["entries"]}),description="분할:")
search_widget=widgets.Text(description="검색:",continuous_update=False,placeholder="파일명 / Enter")
page_widget=widgets.BoundedIntText(value=1,min=1,max=max(1,len(items)),description="이미지:")
prev_button=widgets.Button(description="← 이전")
next_button=widgets.Button(description="다음 →")
delete_button=widgets.Button(description="현재 이미지+라벨 삭제",button_style="danger",layout=widgets.Layout(width="200px"))
points_widget=widgets.Checkbox(value=True,description="관절·연결선",indent=False)
numbers_widget=widgets.Checkbox(value=True,description="번호",indent=False)
boxes_widget=widgets.Checkbox(value=True,description="박스·클래스",indent=False)
info_widget,notice_widget=widgets.Label(),widgets.Label()
output_widget=widgets.Output()
busy=False
shown_entry=None

def refresh(change=None):
    global busy,shown_entry
    if busy:
        return
    busy=True
    try:
        update_items()
        selected=[i for i in items if (split_widget.value=="all" or i["split"]==split_widget.value)
                  and search_widget.value.strip().lower() in i["relative"].lower()]
        page_widget.max=max(1,len(selected))
        page_widget.value=min(page_widget.value,page_widget.max)
        shown_entry=selected[page_widget.value-1] if selected else None
        prev_button.disabled=page_widget.value<=1
        next_button.disabled=page_widget.value>=len(selected)
        delete_button.disabled=shown_entry is None
        info_widget.value=f"검색 {len(selected)}장 / 남은 {len(items)}장 / 삭제 {len(workspace.state['deleted'])}장"
        with output_widget:
            clear_output(wait=True)
            if shown_entry is not None:
                try:
                    workspace.prepare(shown_entry)
                except Exception as exc:
                    print("이미지/라벨 압축 해제 실패:", exc)
                    return
            else:
                workspace.clear_cache()
            show_page(page_widget.value,split_widget.value,search_widget.value,1,
                      points_widget.value,numbers_widget.value,boxes_widget.value)
    finally:
        busy=False

def move(delta):
    page_widget.value=max(1,min(page_widget.max,page_widget.value+delta))

def delete_current(_):
    if busy or shown_entry is None:
        return
    delete_button.disabled=True
    try:
        name=shown_entry["image"]
        workspace.delete(shown_entry)
        notice_widget.value="삭제됨: "+name
    except Exception as exc:
        notice_widget.value="삭제 실패: "+str(exc)
    finally:
        refresh()

def filter_changed(_):
    global busy
    busy=True
    page_widget.value=1
    busy=False
    refresh()

prev_button.on_click(lambda _:move(-1))
next_button.on_click(lambda _:move(1))
delete_button.on_click(delete_current)
for control in (page_widget,points_widget,numbers_widget,boxes_widget):
    control.observe(refresh,names="value")
for control in (split_widget,search_widget):
    control.observe(filter_changed,names="value")
display(widgets.VBox([
    widgets.HBox([split_widget,search_widget]),
    widgets.HBox([prev_button,page_widget,next_button]),
    widgets.HBox([delete_button]),
    widgets.HBox([points_widget,numbers_widget,boxes_widget]),
    info_widget,notice_widget,output_widget]))
refresh()

## 6. 삭제 항목을 제외한 새 ZIP 저장
남은 이미지·라벨을 원본 ZIP에서 파일 하나씩 읽어 새 ZIP에 씁니다. 디스크에 전체 압축 해제하지 않습니다.
새 ZIP은 같은 Drive 폴더에 시각을 붙여 저장합니다. 원본 ZIP은 그대로 유지합니다.
라벨 누락 이미지가 남아 있으면 내보내기가 중단되므로 먼저 검수 화면에서 처리하세요.
원본 메타데이터는 provenance/에 보관하고 최상위 manifest는 남은 accepted 이미지만 포함합니다.

In [ ]:
output_zip=NOTEBOOK_DIR/("cleaned_hand_dataset_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")+".zip")
saved=workspace.export(output_zip)
print("저장:",saved)
print("남은 이미지:",len(workspace.remaining()),"| 삭제:",len(workspace.state["deleted"]))